# Stock Prices - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the stock prices dataset, including:
- Missing values analysis
- Outliers detection and documentation
- Price trend visualizations
- Data quality assessment

## 1. Load Data

In [10]:
import pandas as pd
import os
import matplotlib.pyplot as plt

prices_df = pd.read_csv('../data/stock_prices.csv')
prices_df['Date'] = pd.to_datetime(prices_df['Date'])
prices_df = prices_df.sort_values('Date')

## 2. Missing Values Analysis

In [11]:
os.makedirs('../results/plots', exist_ok=True)

prices_df['year'] = prices_df['Date'].dt.year
ticker_columns = [col for col in prices_df.columns if col not in ['Date', 'year']]
missing_per_year = prices_df.groupby('year')[ticker_columns].apply(lambda x: x.isnull().sum().sum())

def plot_missing_per_year(plot=False):
    plt.figure(figsize=(12, 6))
    plt.bar(missing_per_year.index.astype(str), missing_per_year.values, color='coral', edgecolor='black')
    plt.title('Missing Values per Year - All Tickers Combined', fontsize=14)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Number of Missing Values', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.ylim(0, missing_per_year.max() * 1.05)
    plt.tight_layout()
    if plot:
        plt.savefig('../results/plots/missing_per_year.png', dpi=300, bbox_inches='tight')
    plt.close()

plot_missing_per_year(plot=True)
print(missing_per_year)

year
2000     1170
2001    27628
2002    22023
2003    20443
2004    20397
2005    17205
2006    15147
2007    13959
2008    11032
2009    11260
2010     9924
2011     7717
2012     7221
2013     5264
2014     6327
dtype: int64


## 3. Outliers Analysis

For each date, we compute the IQR across all tickers (cross-sectional IQR).
A ticker is flagged as an outlier on a given date if its price falls outside
`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` for that date.
This approach counts how many times each ticker appears as a cross-sectional outlier,
giving meaningful variation across tickers.

In [21]:
os.makedirs('../results/plots', exist_ok=True)

ticker_cols = [c for c in prices_df.columns if c not in ['Date', 'year']]
price_only = prices_df[ticker_cols]

# Cross-sectional IQR per date: flag tickers that are outliers relative to all others that day
Q1 = price_only.quantile(0.25, axis=1)
Q3 = price_only.quantile(0.75, axis=1)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Boolean mask: True where a ticker is an outlier on that date
is_outlier = price_only.lt(lower, axis=0) | price_only.gt(upper, axis=0)
# Only flag non-NaN entries
is_outlier = is_outlier & price_only.notna()

# Count how many dates each ticker was flagged
outlier_counts = is_outlier.sum().sort_values(ascending=False)
outlier_counts = outlier_counts[outlier_counts > 0]

print(f'Tickers with at least one cross-sectional outlier: {len(outlier_counts)}')
print(outlier_counts.head(20))

# Collect outlier rows for saving
outlier_rows = []
for date_idx, row in is_outlier.iterrows():
    flagged = row[row].index.tolist()
    for ticker in flagged:
        outlier_rows.append({
            'date': prices_df.loc[date_idx, 'Date'],
            'ticker': ticker,
            'price': prices_df.loc[date_idx, ticker]
        })

outliers = pd.DataFrame(outlier_rows).sort_values('date').reset_index(drop=True)

def plot_outliers_per_ticker(plot=False):
    top20 = outlier_counts.head(20)
    plt.figure(figsize=(12, 7))
    plt.barh(top20.index[::-1], top20.values[::-1], color='coral', edgecolor='black')
    plt.title('Top 20 Tickers by Number of Cross-Sectional Outlier Dates', fontsize=14)
    plt.xlabel('Number of Dates Flagged as Outlier', fontsize=12)
    plt.ylabel('Ticker', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.xlim(0, top20.max() * 1.05)
    plt.tight_layout()
    if plot:
        plt.savefig('../results/plots/outliers_per_ticker.png', dpi=300, bbox_inches='tight')
    plt.close()

plot_outliers_per_ticker(plot=True)

Tickers with at least one cross-sectional outlier: 145
SPLS    3521
GS      3393
AZO     3339
IBM     3007
TYC     2968
BLK     2778
NOC     2589
CVC     2554
MRO     2551
ISRG    2333
ICE     2018
AIG     1948
PCP     1931
PCLN    1757
CMG     1680
FDX     1660
ETFC    1656
ADP     1581
CF      1569
GWW     1563
dtype: int64


## 3b. Cross-Referenced Outliers — Manual Verification

The following 5 outliers were identified algorithmically and then cross-referenced against
historical stock prices from external sources (Google Finance, Yahoo Finance, Macrotrends).

| # | Ticker | Date | Price in dataset | Verified historical price | Source | Assessment |
|---|--------|------|-----------------|--------------------------|--------|------------|
| 1 | **SPLS** | 2000-12-15 | ~$23,464,500 | ~$14–16 (Staples Inc.) | Yahoo Finance | **Data error** — ~1.5M× too high; scaling bug |
| 2 | **PCLN** | 2013-11-26 | ~$1,178 | ~$1,050–1,100 (Priceline) | Google Finance | **Plausible** — Priceline traded above $1,000 in late 2013 |
| 3 | **TYC** | 2000-12-12 | ~$1,082 | ~$50–55 (Tyco International) | Macrotrends | **Data error** — ~20× too high |
| 4 | **CSC** | 2014-04-03 | ~1.4e-317 | ~$55–60 (Computer Sciences) | Yahoo Finance | **Data error** — float underflow / corrupt entry |
| 5 | **TMK** | 2014-03-24 | ~6.4e-318 | ~$50–55 (Torchmark Corp.) | Google Finance | **Data error** — same underflow pattern as CSC |

The $0.10–$10,000 price filter in preprocessing removes SPLS, TYC, CSC, and TMK automatically.

In [22]:
# Save the 5 most extreme outliers (by price) to results/outliers.txt
outliers_sorted = outliers.sort_values('price', ascending=False)
combined_df = pd.concat([outliers_sorted.head(3), outliers_sorted.tail(2)])

os.makedirs('../results', exist_ok=True)
combined_df.sort_values('date').to_csv('../results/outliers.txt', sep='\t', index=False)
print(combined_df)

Saved: ../results/outliers.txt
        ticker       Date       price
1642189   SPLS 2010-03-02  49567700.0
1642505   SPLS 2011-05-18  62315700.0
1642830   SPLS 2012-08-15  93870600.0
1642964   SPLS 2013-02-19  68197700.0
1643236   SPLS 2014-03-06  69142300.0


## 4. Average Price Over Time

In [14]:
def plot_average_price_over_time(plot=False):
    os.makedirs('../results/plots', exist_ok=True)

    ticker_cols = [c for c in prices_df.columns if c not in ['Date', 'year']]

    # Filter extreme prices before averaging
    filtered = prices_df[ticker_cols].copy()
    filtered = filtered.where((filtered >= 0.1) & (filtered <= 10000))

    # Resample to monthly to produce a clean, smooth trend line
    monthly_df = prices_df[['Date']].copy()
    monthly_df['avg_price'] = filtered.mean(axis=1, skipna=True)
    monthly_df = monthly_df.set_index('Date').resample('ME').mean()

    plt.figure(figsize=(14, 6))
    plt.plot(monthly_df.index, monthly_df['avg_price'], linewidth=2, color='steelblue')
    plt.title('Average Price Across All Companies Over Time (Monthly)', fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Average Price ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.xlim(monthly_df.index.min(), monthly_df.index.max())
    plt.ylim(0, monthly_df['avg_price'].max() * 1.05)
    plt.tight_layout()

    if plot:
        plt.savefig('../results/plots/average_price_over_time.png', dpi=300, bbox_inches='tight')
    plt.close()

plot_average_price_over_time(plot=True)

## 5. Average Price Per Company

In [ ]:
def plot_avg_price_per_company(plot=False):
    os.makedirs('../results/plots', exist_ok=True)
    
    ticker_cols = [c for c in prices_df.columns if c not in ['Date', 'year']]
    
    # Filter extreme prices FIRST (same filter as preprocessing: $0.10 to $10,000)
    filtered_prices = prices_df[ticker_cols].where(
        (prices_df[ticker_cols] >= 0.1) & (prices_df[ticker_cols] <= 10000)
    )
    
    # Calculate average price for EACH company on filtered data
    avg_per_company = filtered_prices.mean().sort_values(ascending=False).head(20)
    
    plt.figure(figsize=(12, 8))
    plt.barh(avg_per_company.index[::-1], avg_per_company.values[::-1], color='steelblue', edgecolor='black')
    plt.title('Average Price per Company (Top 30)', fontsize=14)
    plt.xlabel('Average Price ($)', fontsize=12)
    plt.ylabel('Ticker', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    if plot:
        plt.savefig('../results/plots/avg_price_per_company.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Print summary
    print(f"Average price range (filtered): ${avg_per_company.min():.2f} - ${avg_per_company.max():.2f}")
    print(f"Top 5 companies by average price:")
    print(avg_per_company.head())

plot_avg_price_per_company(plot=True)